Предмет "Искусственный интеллект"

Лабораторная работа №2

Грибушенков Михаил, группа РФЗ-2-2022

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

# defines
filename = "Mine_Dataset.xls"

In [3]:
db = pd.read_excel (filename, sheet_name=1) # Загрузка базы данных из файла
db.head(30)

,V,H,S,M
0,0.338157,0.000000,0.0,1
1,0.320241,0.181818,0.0,1
2,0.287009,0.272727,0.0,1
3,0.256284,0.454545,0.0,1
4,0.262840,0.545455,0.0,1
5,0.240966,0.727273,0.0,1
6,0.254410,0.818182,0.0,1
7,0.234924,1.000000,0.0,1
8,0.353474,0.000000,0.6,1
9,0.335347,0.181818,0.6,1


In [14]:
# Деление набора данных на тестовый и обучающий, статистика сего.
from sklearn.model_selection import train_test_split
import scipy.stats as stats

features = db.drop(["M"], axis=1)
target = db["M"]
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.30, random_state=42)
stats.ttest_ind (a=target_train, b=target_test)


TtestResult(statistic=np.float64(-0.6528416441682411), pvalue=np.float64(0.5143047972923565), df=np.float64(336.0))

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Список изучаемых моделей
models = []
models.append(("Логистическая регрессия", LogisticRegression(solver='lbfgs', max_iter=5000)))
models.append(("LDA", LinearDiscriminantAnalysis()))
models.append(("Ка ближайших соседев", KNeighborsClassifier()))
models.append(("Дерево решений", DecisionTreeClassifier()))
models.append(("NB", GaussianNB()))
models.append(("SVC", SVC(gamma='auto')))

# Их метрики
results = []
model_names = []
for name, model in models:
  kfold = StratifiedKFold(n_splits=10, random_state=1, shuffle=True)
  cv_results = cross_val_score(model, features_train, target_train, cv=kfold, scoring='accuracy')
  results.append(cv_results)
  model_names.append(name)
  print('%s: %f (%f)' % (name, cv_results.mean(), cv_results.std()))

Логистическая регрессия: 0.436957 (0.075991)
LDA: 0.423370 (0.062357)
Ка ближайших соседев: 0.365036 (0.054524)
Дерево решений: 0.525362 (0.078966)
NB: 0.440036 (0.104673)
SVC: 0.356159 (0.061553)


Видно, что дерево работает лучше всего.

In [26]:
# Обучим дерево
tree = DecisionTreeClassifier(max_depth=6)
tree.fit(features_train, target_train)
tree_prediction_result = tree.predict(features_test)
tree_accuracy = accuracy_score(target_test, tree_prediction_result)

print(f"Tree accuracy: {tree_accuracy}")
print(f"Tree prediction: {tree_prediction_result}")

Tree accuracy: 0.5
Tree prediction: [4 3 1 2 3 3 1 2 3 1 3 1 3 2 4 2 3 3 2 1 4 5 1 2 4 3 5 3 5 2 3 4 3 5 2 4 1
 3 2 4 1 1 3 3 3 2 5 5 1 5 2 2 4 4 2 4 1 1 1 4 4 4 5 3 3 1 4 4 5 1 4 2 4 2
 5 5 1 2 2 2 3 1 1 1 4 1 5 3 4 2 5 4 1 4 2 4 1 2 1 2 5 5]


In [27]:
predicted = tree_prediction_result

from sklearn import metrics

recall = metrics.recall_score(target_test, predicted, average="weighted")
precision = metrics.precision_score(target_test, predicted, average="weighted")
print("Recall (all 1s predicted right):", round(recall,3))
print("Precision (confidence when predicting a 1):", round(precision,3))
print("Detail:")
print(metrics.classification_report(target_test, predicted, target_names=[str(i) for i in np.unique(target_test)]))


Recall (all 1s predicted right): 0.5
Precision (confidence when predicting a 1): 0.475
Detail:
              precision    recall  f1-score   support

           1       0.65      0.71      0.68        21
           2       0.77      0.85      0.81        20
           3       0.30      0.33      0.32        18
           4       0.45      0.48      0.47        21
           5       0.20      0.14      0.16        22

    accuracy                           0.50       102
   macro avg       0.48      0.50      0.49       102
weighted avg       0.48      0.50      0.49       102



Видно, что модель лучше всего распознает противотанковые мины, соответствующие индексу 2. Прочие мины, как видно из визуализаций в ЛР 1, мало отличаются в признаках, соответственно, слабо выделяются.